# Quant CSOT — Week 3 Challenge

Complete the `get_weights` function below, then run the backtest cells to
check your strategy against the portfolio constraints and see its Sharpe
ratio. See `docs/competition_rules.md`, `docs/evaluation.md`, and
`docs/submission_format.md` for the full task description.

**Before running:** download `features.parquet`, `universe.parquet`, and
`returns.parquet` into the `data/` folder. Instructions are in
`data/download_data.md`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT / "src"))

import utils


## 1. Load data

In [ ]:
features, universe, returns = utils.load_data()

print("features:", features.shape, " (rows=dates, columns=(feature, stock_id))")
print("universe:", universe.shape)
print("returns: ", returns.shape)


In [ ]:
features.head()


## 2. Implement your strategy

`get_weights` is called once per trading day. It receives:

- `features`: every feature's full history **strictly before** today (no look-ahead)
- `today_universe`: a 0/1 Series telling you which stocks can be traded today

It must return a `dict[str, float]` mapping stock id -> weight, containing
**only** stock ids where `today_universe` is `1`.

Keep the portfolio constraints in mind (see `docs/competition_rules.md`):
- `sum(|w|) <= 1 + 1e-4`
- `|w| <= 0.1` per stock
- `|sum(w)| <= 1e-4` (dollar neutral)


In [ ]:
def get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]:
    """
    Calculate stock weights for the portfolio on the current trading day.

    Parameters
    ----------
    features : pd.DataFrame
        MultiIndex columns (feature_name, stock_id). Index is the chronological
        date history strictly before the current trading day.
    today_universe : pd.Series
        Index: stock identifiers. Values: 0 or 1, where 1 means the stock is
        tradable today.

    Returns
    -------
    dict[str, float]
        Keys: stock identifiers (only those with today_universe == 1).
        Values: portfolio weights for the current trading day.
    """
    # TODO: replace this with your actual signal + portfolio construction.
    # Placeholder below: a trivial equal-weight book that longs the first
    # half of today's tradable universe and shorts the rest, with each leg
    # sized so gross exposure on each side is exactly equal (dollar neutral)
    # regardless of how the universe splits. It satisfies every constraint
    # but carries no real alpha — just here so the backtest runs end to end.
    tradable = today_universe[today_universe == 1].index.tolist()
    n = len(tradable)
    if n < 2:
        return {s: 0.0 for s in tradable}

    half = n // 2
    longs = tradable[:half]
    shorts = tradable[half:]

    max_gross = min(1.0, 0.1 * min(len(longs), len(shorts)) * 2)  # respect unit-capital + max-weight
    w_long = (max_gross / 2) / len(longs)
    w_short = (max_gross / 2) / len(shorts)

    weights = {s: w_long for s in longs}
    weights.update({s: -w_short for s in shorts})
    return weights


## 3. Backtest

In [ ]:
weights = utils.backtest_strategy(get_weights, features, universe)
weights.head()


## 4. Validate constraints

In [ ]:
checks = utils.validate_weights(weights, universe)
for name, passed in checks.items():
    print(f"{name:20s} {'PASS' if passed else 'FAIL'}")


## 5. Score the strategy (training period only)

In [ ]:
metrics = utils.summarize_performance(weights, returns)
for k, v in metrics.items():
    print(f"{k:16s} {v:.4f}")


In [ ]:
net_pnl = utils.net_pnl(weights, returns)
net_pnl.cumsum().plot(figsize=(10, 4), title="Cumulative Net PnL")
plt.show()


## 6. Export submission

`submission.csv` is saved to the repo root and must have the same shape and
index as `data/universe.parquet`. Run `src/validate_submission.py` from the
repo root to double check before you submit (see `docs/submission_format.md`).


In [ ]:
output_path = REPO_ROOT / "submission.csv"
weights.to_csv(output_path)
print("Saved", output_path, "with shape", weights.shape)
